#DAY 11 Databricks Challenge

##Challenges
### 🛠️ Tasks:

1. Calculate statistical summaries
2. Test hypotheses (weekday vs weekend)
3. Identify correlations
4. Engineer features for ML

####Task 1 - Calculate statistical summaries - Detects outliers, check skewness, validates assumptions

| metric    | meaning                   |
| --------- | ------------------------- |
| count     | number of non-null values |
| mean      | average price             |
| stddev    | price variation           |
| min / max | price range               |





In [0]:
#loading into events df
events = spark.read.table("workspace.default.silver_events_part")

In [0]:
events.describe(["price"]).show()


#**************************************************

####Task 2 - Hypothesis Testing (Weekday vs Weekend) - How do users behave during weekday/weekend.
##### Sunday=1, Saturday=7

#####Step 1: Create weekend flag (Fixing your logic)

In [0]:
from pyspark.sql import functions as F

events_flagged = events.withColumn(
    "is_weekend",
    F.dayofweek("event_time").isin([1, 7])  # Sunday=1, Saturday=7
)

#####Step 2: Compare behavior

In [0]:
events_flagged.groupBy("is_weekend", "event_type") \
    .count() \
    .orderBy("is_weekend", "event_type") \
    .show()


#**************************************************

####Task 3 - Correlation Analysis (Relationships Between Variables)
##### Correlation measures linear relationship strength between two numeric columns.
- Values range:
- +1 → strong positive
- 0 → no relationship
- -1 → strong negative

In [0]:
events.stat.corr("price", "user_id") 
#Correlation only works on numeric, row-level columns

#**************************************************

####Task 4 - Feature Engineering (Most Important for ML)
#####Feature engineering transforms raw data into model-ready signals.

#####Feature 1: Time based features

In [0]:
#printing schema to verify
events.printSchema()


In [0]:
#creating new features
from pyspark.sql import functions as F

features = events \
    .withColumn("hour", F.hour("event_time")) \
    .withColumn("day_of_week", F.dayofweek("event_time"))


In [0]:
#viewing features
features.select("event_time", "hour", "day_of_week").show(5, truncate=False)

#####Feature 2 - Time since first event

In [0]:
#Creating feature logic
from pyspark.sql import Window

window = Window.partitionBy("user_id") \
               .orderBy("event_time") \
               .rowsBetween(Window.unboundedPreceding, Window.currentRow)

features = features.withColumn(
    "first_event_time",
    F.first("event_time").over(window)
)


In [0]:
#Computing the feature
features = features.withColumn(
    "time_since_first_view",
    F.unix_timestamp("event_time") -
    F.unix_timestamp("first_event_time")
)


In [0]:
#Viewing the new feature output
features.select(
    "user_id",
    "event_time",
    "first_event_time",
    "time_since_first_view"
).show(10, truncate=False)



%md
## **For me more such learning and insights in**
- ### [LinkedIn](https://www.linkedin.com/in/ilakkiyan-av/) 
- ### [Youtube](https://www.youtube.com/@ilakkiyanav) 